# cancer-imaging radiomics: feature reproducibility + outcome colab

a radiomic feature is only a biomarker if it both tracks the outcome and stays stable under the upstream variability that produced it. this is the downstream half of a two-project pipeline (project 1 = upstream mri reconstruction): segmentation -> radiomic features -> two checks, (1) feature reproducibility under a +/-1 voxel segmentation perturbation with icc(2,1) including the -300 hu parenchyma-floor leakage fix, and (2) feature->outcome correlation with a volume-confound check, qc throughout

> the whole synthetic pipeline is numpy-only and runs in seconds on a **cpu** runtime, no gpu and no download needed. a gpu is only for the monai segmenter (build plan) and the real lidc run

> colab storage is ephemeral mount drive and save outputs there
| accelerator | good for | notes |
|---|---|---|
| a100 / h100 | both projects fast | ampere/hopper tf32 + bf16 ideal for the unrolled net |
| l4 | both projects | ada 24 gb bf16/tf32 great value |
| t4 (free) | synthetic + small fastmri | 16 gb use amp_dtype fp16 (no bf16) |
| g4 / older gpu | synthetic + small runs | auto-detected via nvidia-smi |
| cpu | smoke test + synthetic | validates the pipeline too slow for real training |
| v5e-1 / v6e-1 tpu | project 2 only (maybe) | see the tpu note below |

> tpu note project 1 reconstruction is built on complex-valued ffts not reliably
> supported on tpu/xla use a gpu runtime for project 1 project 2 (real-valued
> segmentation) can in principle use a tpu but monai + torch_xla is finicky a gpu
> is still the smoother path

## 1. runtime

In [ ]:
import os, subprocess, sys
print("Python:", sys.version.split()[0])
try:
    print("GPU:", subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip())
except Exception:
    print("No NVIDIA GPU (CPU/TPU runtime).")

## 2. get the project code

set SETUP to github drive or upload (same as project 1)

In [ ]:
SETUP = "github"   # "github" | "drive" | "upload"
REPO_URL = "https://github.com/cosmicquilt/segmentation-radiomics.git"
PROJECT_DIR = "/content/segmentation-radiomics"

import os
if SETUP == "github":
    if os.path.exists(PROJECT_DIR):
        get_ipython().system(f'git -C {PROJECT_DIR} pull')   # already cloned, pull latest
    else:
        get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
elif SETUP == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/job/projects/02-segmentation-radiomics"  # set me
elif SETUP == "upload":
    from google.colab import files
    import zipfile, io
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall("/content")
    PROJECT_DIR = "/content/" + name.replace(".zip", "")

os.chdir(PROJECT_DIR)
import sys
SRC = os.path.join(PROJECT_DIR, "src")
sys.path.insert(0, SRC)                          # in-kernel imports
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")  # for !python cells
print("project dir:", os.getcwd())

## 3. install + smoke test + unit tests (numpy core)

In [ ]:
!pip -q install pyyaml pytest
!python scripts/smoke_test.py
!python -m pytest tests/ -q

## 4. run the synthetic pipeline end-to-end

In [ ]:
!python -m seg_radiomics.cli run --config configs/default.yaml

**what to check in the output above**
- segmentation Dice ~1.0 (the synthetic nodule is threshold-separable, high by construction not by skill)
- reproducibility table: median ICC jumps from ~0.39 raw to ~0.95 after the -300 HU floor (the leakage fix)
- the low-signal line flags firstorder_StdDev (near-constant across the synthetic cohort, so its ICC is ill-conditioned)
- the volume-confound block flags firstorder_Energy (its high AUC is largely a lesion-size proxy)

## 5. the hero figure (raw vs floored ICC per feature)

In [ ]:
!python scripts/make_figures.py
from IPython.display import Image
Image('docs/figures/parenchyma_floor_recovery.png')

## 6. real data lidc-idri (real malignancy labels)

the lidc loader is already written (data/lidc.py) it returns the consensus nodule mask and the radiologist malignancy rating so the feature->outcome correlation runs against a real label instead of a manufactured one grab a subset of scans (not the full 133 gb) save to drive see scripts/download_data.md

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
LIDC = '/content/drive/MyDrive/lidc'
os.makedirs(LIDC, exist_ok=True)

In [ ]:
!pip -q install pylidc pandas tcia_utils
# download ~80 lidc series into drive (once):
# from tcia_utils import nbia
# nbia.downloadSeries(nbia.getSeries(collection='LIDC-IDRI')[:80], path=LIDC)

In [ ]:
# one-time pylidc config pointing at the dicoms then run the pipeline
cfg = f"[dicom]\npath = {LIDC}/LIDC-IDRI\nwarn = True\n"
open(os.path.expanduser('~/.pylidcrc'), 'w').write(cfg)
!python -m seg_radiomics.cli run --config configs/lidc.yaml

## accelerator notes
- synthetic pipeline cpu is fine
- monai 3d unet use a gpu (t4 ok a100/l4 faster) bf16 autocast on ampere+
- tpu possible for the real-valued segmenter but monai + torch_xla is rough prefer a gpu